# Joint disease-conditioned GCN prioritization

This notebook trains one shared GCN encoder across all diseases. Each training sample supplies the same PPI graph, a disease-specific seed indicator, and a disease ID. A learned disease embedding is combined with every gene representation before producing one score per gene.

Known genes are split into outer training and held-out test sets. Only outer-training genes can become visible seeds or positive labels; held-out genes are also excluded from sampled negatives.

In [18]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if not (project_root / 'bioGraph').is_dir():
    raise FileNotFoundError('Start Jupyter from the repository root or notebooks directory.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd

from bioGraph.data.loading import load_disease_genes, load_ppi_graph
from bioGraph.gcn_prioritization import predict_from_seed_genes, train_all_diseases

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load the graph and disease associations

The processed subgraph keeps this example practical on a laptop. Replace `ppi_path` with `data/raw/PPI202207.txt` to train on the complete PPI network.

In [19]:
ppi_path = project_root / 'data' / 'processed' / 'subgraph_5377.txt'
disease_path = project_root / 'data' / 'raw' / 'pcbi.1004120.s004.txt'

graph = load_ppi_graph(ppi_path)
diseases = load_disease_genes(disease_path)

print(f'Graph: {graph.number_of_nodes():,} genes, {graph.number_of_edges():,} interactions')
print(f'Diseases: {len(diseases)}')

Graph: 5,377 genes, 94,987 interactions
Diseases: 70


## Train one model jointly across all diseases

The returned model contains one shared encoder and one disease-embedding table. There is no encoder pretraining and no per-disease fine-tuning. `keep_details=True` retains rankings and score tensors for inspection below.

In [23]:
result = train_all_diseases(
    graph,
    diseases,
    k_values=(25, 300),
    hidden_dim=64,
    disease_embedding_dim=16,
    epochs=100,
    learning_rate=0.01,
    weight_decay=1e-4,
    negative_ratio=5,
    train_fraction=0.75,
    inner_seed_fraction=2/3,
    seed=0,
    task_batch_size=16,
    keep_details=True,
)

model = result['model']
disease_results = result['disease_results']
print(f"Finished {len(result['losses'])} epochs on {result['device']}")
print(f"Final pairwise loss: {result['losses'][-1]:.4f}")

Epoch   1/100: pairwise loss=0.7030
Epoch   2/100: pairwise loss=0.7098
Epoch   3/100: pairwise loss=0.6925
Epoch   4/100: pairwise loss=0.6932
Epoch   5/100: pairwise loss=0.6896
Epoch   6/100: pairwise loss=0.6862
Epoch   7/100: pairwise loss=0.6919
Epoch   8/100: pairwise loss=0.6908
Epoch   9/100: pairwise loss=0.6838
Epoch  10/100: pairwise loss=0.6875
Epoch  11/100: pairwise loss=0.6657
Epoch  12/100: pairwise loss=0.6809
Epoch  13/100: pairwise loss=0.6747
Epoch  14/100: pairwise loss=0.6675
Epoch  15/100: pairwise loss=0.6457
Epoch  16/100: pairwise loss=0.6368
Epoch  17/100: pairwise loss=0.6520
Epoch  18/100: pairwise loss=0.6261
Epoch  19/100: pairwise loss=0.6112
Epoch  20/100: pairwise loss=0.6244
Epoch  21/100: pairwise loss=0.6158
Epoch  22/100: pairwise loss=0.6126
Epoch  23/100: pairwise loss=0.6096
Epoch  24/100: pairwise loss=0.5903
Epoch  25/100: pairwise loss=0.5833
Epoch  26/100: pairwise loss=0.6139
Epoch  27/100: pairwise loss=0.6061
Epoch  28/100: pairwise loss

## Inspect held-out performance

Each row is computed against that disease's held-out test genes. Training genes are excluded from its final candidate ranking.

In [24]:
metric_rows = [
    {'disease': name, **details['metrics']}
    for name, details in disease_results.items()
]
metrics_by_disease = pd.DataFrame(metric_rows).set_index('disease')
display(metrics_by_disease)
display(metrics_by_disease.mean().rename('mean across diseases'))

,recall@25,ap@25,recall@300,ap@300
disease,,,,
adrenal gland diseases,0.000000,0.000000,0.000000,0.000000
alzheimer disease,0.142857,0.047619,0.285714,0.056277
amino acid metabolism inborn errors,0.307692,0.071357,0.692308,0.111054
amyotrophic lateral sclerosis,0.000000,0.000000,0.000000,0.000000
anemia aplastic,0.600000,0.240000,1.000000,0.285513
...,...,...,...,...
spondylarthropathies,0.250000,0.062500,0.250000,0.062500
tauopathies,0.000000,0.000000,0.111111,0.001984
uveal diseases,0.000000,0.000000,0.250000,0.001250


recall@25     0.126221
ap@25         0.049349
recall@300    0.288584
ap@300        0.053867
Name: mean across diseases, dtype: float64

## Inspect one disease and run a conditioned query

Inference must use the disease ID that selects the learned embedding. The query ranking excludes the supplied seed genes.

In [25]:
disease_name = 'breast neoplasms'
details = disease_results[disease_name]
display(pd.Series(details['metrics'], name=disease_name))
display(pd.DataFrame(details['ranking'][:10]))

query_seed_genes = details['train_genes'][:10]
query_ranking = predict_from_seed_genes(
    model,
    result['graph_data'],
    query_seed_genes,
    disease_id=result['disease_to_id'][disease_name],
)
pd.DataFrame(query_ranking[:20])

recall@25     0.000000
ap@25         0.000000
recall@300    0.200000
ap@300        0.001814
Name: breast neoplasms, dtype: float64

,gene_id,symbol,score
0,1457,CSNK2A1,5.316484
1,6714,SRC,5.284867
2,7157,TP53,5.252084
3,5566,PRKACA,5.187221
4,162998,OR7D2,5.161602
5,5578,PRKCA,4.805682
6,5594,MAPK1,4.619444
7,5595,MAPK3,4.460785
8,2932,GSK3B,4.253034
9,983,CDK1,4.233430


,gene_id,symbol,score
0,6714,SRC,5.139151
1,5566,PRKACA,5.105847
2,7157,TP53,5.034921
3,1457,CSNK2A1,4.965474
4,5578,PRKCA,4.718586
5,5594,MAPK1,4.448475
6,5595,MAPK3,4.324045
7,2932,GSK3B,4.175141
8,1017,CDK2,4.101466
9,983,CDK1,4.076454
